<a href="https://colab.research.google.com/github/zohaybhassan/StressGuard-ML-Based-Stress-Detection/blob/main/Data%20Ingestion%20%26%20Tabular%20Foundation%20Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- PHASE 1: LOADING & CLEANING ---

# 1. Load the CSV
file_path = 'expanded_sleep_health_dataset.csv'
df = pd.read_csv(file_path)
print(f"Initial Dataset Shape: {df.shape}")

# 3. Isolate the Target (Stress Level 1-10)
target_col = 'Stress Level'
X = df.drop(target_col, axis=1)
y = df[target_col]

print(f"Current Stress Level counts:\n{y.value_counts().sort_index()}")

# --- PHASE 2: ENCODING & SCALING ---

# 4. One-Hot Encode Categorical (Text) Variables
# This turns 'Gender_Male' into a 1 or 0 so the math works
categorical_cols = ['Gender', 'Occupation', 'BMI Category', 'Sleep Disorder']
cols_to_encode = [col for col in categorical_cols if col in X.columns]

X_encoded = pd.get_dummies(X, columns=cols_to_encode, drop_first=True)
print(f"\nCategorical columns encoded: {cols_to_encode}")

# 5. Z-Score Normalization for Numerical Variables
# This ensures a Heart Rate of 80 and a Step Count of 8000 are weighted equally
numerical_cols = ['Age', 'Sleep Duration', 'Quality of Sleep', 'Physical Activity Level', 'Heart Rate', 'Daily Steps']
cols_to_scale = [col for col in numerical_cols if col in X_encoded.columns]

scaler = StandardScaler()
X_encoded[cols_to_scale] = scaler.fit_transform(X_encoded[cols_to_scale])

print(f"Final Input Shape for ML: {X_encoded.shape}")

# Peek at the final cleaned features
X_encoded.head()

Initial Dataset Shape: (1500, 8)
Current Stress Level counts:
Stress Level
1      74
2      54
3     116
4     145
5     215
6     214
7     236
8     207
9     149
10     90
Name: count, dtype: int64

Categorical columns encoded: ['Gender', 'Occupation', 'BMI Category']
Final Input Shape for ML: (1500, 22)


,Age,Heart Rate,Daily Steps,Sleep Duration,Gender_Male,Occupation_Artist,Occupation_Chef,Occupation_Doctor,Occupation_Engineer,Occupation_Lawyer,...,Occupation_Sales Representative,Occupation_Salesperson,Occupation_Scientist,Occupation_Software Engineer,Occupation_Student,Occupation_Teacher,Occupation_Writer,BMI Category_Obese,BMI Category_Overweight,BMI Category_Underweight
0,-0.407320,-1.533797,1.189356,-0.834831,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,-0.352237,-0.634609,1.609261,-1.167994,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
2,-0.186988,-1.206819,-0.385026,1.053090,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,-1.343729,1.817722,-1.597969,0.275711,True,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
4,-1.674226,-0.307631,0.161029,1.164144,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,True,False


In [11]:
from imblearn.over_sampling import SMOTE
import pandas as pd

# --- PHASE 3: APPLY SMOTE ---

print("Original Stress Distribution :")
print(y.value_counts().sort_index())

# 1. Initialize SMOTE
# random_state=42 ensures you get the exact same synthetic data every time
smote = SMOTE(random_state=42)

# 2. Fit and resample the encoded features (X_encoded) and target (y)
X_balanced, y_balanced = smote.fit_resample(X_encoded, y)

# 3. Verify the new distribution
print("\nBalanced Stress Distribution (After SMOTE):")
print(y_balanced.value_counts().sort_index())

print(f"\nNew Balanced Input Shape for ML: {X_balanced.shape}")
# Safely convert ONLY the True/False columns to 1s and 0s
bool_cols = X_balanced.select_dtypes(include=['bool']).columns
X_balanced[bool_cols] = X_balanced[bool_cols].astype(int)

# Check the first few rows to verify it worked without ruining the decimals
X_balanced.head()
# --- EXPORT FINAL ITERATION 1 DATASET ---

# 4. Reattach the balanced target to the balanced features
final_balanced_df = X_balanced.copy()
final_balanced_df['Target_Stress_Level'] = y_balanced.values
# Convert all True/False boolean columns to 1s and 0s for a cleaner presentation
X_balanced = X_balanced.astype(int)
# 5. Export to CSV
export_filename = 'StressGuard_Iteration1_Balanced_Data.csv'
final_balanced_df.to_csv(export_filename, index=False)

print(f"\nData Pipeline Complete.")
print(f"Perfectly balanced dataset saved as: {export_filename}")

Original Stress Distribution :
Stress Level
1      74
2      54
3     116
4     145
5     215
6     214
7     236
8     207
9     149
10     90
Name: count, dtype: int64

Balanced Stress Distribution (After SMOTE):
Stress Level
1     236
2     236
3     236
4     236
5     236
6     236
7     236
8     236
9     236
10    236
Name: count, dtype: int64

New Balanced Input Shape for ML: (2360, 22)

Data Pipeline Complete.
Perfectly balanced dataset saved as: StressGuard_Iteration1_Balanced_Data.csv
